Use this to familiarize yourself with the code in this repo

Run stacking simulator with rule based agent

In [ ]:
# ============================================================
# VISUALIZE HIERARCHICAL RULE-BASED AGENT — SINGLE SEED
# Interactive step-by-step viewer
# Change EVAL_SEED below to inspect any episode
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from stack.envs.stack_gym import StackEnv
from stack.agents.hierarchical_rule_based_agent import HierarchicalAgent

# ============================================================
# CONFIG
# ============================================================

config_dict = {
    "vessel_shape": (10, 8, 5),
    "yard_shape": (10, 9, 5),
    "num_containers": 400,
    "group_num": 10,
    "group_placement": "random",
    "observation_type": "flat_parsed",
    "reward_norm": False,
    "reward_clip": False,
    "stack_fill_penalty": True,
    "container_sizes": True,
    "enable_imo": True,
    "random_group_sizes": True
}

EVAL_SEED = 21   # ← change to inspect any seed
MAX_STEPS = 500

# ============================================================
# CREATE ENV + AGENT
# ============================================================

# Change policy_type below to test other heurustic policies
# like "rule_based_grouped", "rule_based" or "random"
# defined in agents/hierarchical_rule_based_agent.py
env = StackEnv(config=config_dict, render_mode="rgb_array")
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based_grouped",
    low_level_policy_type="rule_based_grouped",
)

# ============================================================
# RUN EPISODE & COLLECT FRAMES
# ============================================================

observation, info = env.reset(seed=EVAL_SEED)
total_reward = 0.0
step_count = 0
frames = []  # (img, step, action, selected_bay, reward, cum_reward)

for _ in range(MAX_STEPS):
    valid_actions = env._get_valid_yard_actions()
    if len(valid_actions) == 0:
        print("No valid actions remaining. Episode finished!")
        break
    try:
        action, action_info = agent.get_action(observation, valid_actions)
        if action is None:
            print("Agent returned None. Episode finished!")
            break
    except Exception as e:
        print(f"Agent error: {e}")
        break

    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1

    # Toggle enhanced_visibility to True for better visibility of the environment
    img = env.render(enhanced_visibility=False) 
    selected_bay = action_info.get("selected_bay") if action_info else None
    frames.append((img, step_count, int(action), selected_bay, reward, total_reward))

    if terminated or truncated:
        break

env.close()
print(f"Seed {EVAL_SEED} | Steps: {step_count} | Total Reward: {total_reward:.4f}")


In [ ]:
# ============================================================
# INTERACTIVE VIEWER WIDGET (with Save Button)
# ============================================================

slider = widgets.IntSlider(
    value=0, min=0, max=len(frames) - 1, step=1,
    description="Step:", continuous_update=False,
    layout=widgets.Layout(width="60%"),
)
prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
save_btn = widgets.Button(description="💾 Save", layout=widgets.Layout(width="80px"), button_style="info")
info_label = widgets.HTML()
out = widgets.Output()
save_status = widgets.HTML()

def save_current_frame(idx):
    img, step, action, selected_bay, reward, cum_reward = frames[idx]
    import os
    os.makedirs("saved_eval_frames", exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(30, 16))
    ax.imshow(img)

    ax.axis("off")
    fig.tight_layout()
    
    save_path = f"saved_eval_frames/hrl_seed{EVAL_SEED}_step_{step}.png"
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    
    save_status.value = f'<span style="color:green;"><b>✓ Saved to {save_path}</b></span>'

def show_frame(idx):
    img, step, action, selected_bay, reward, cum_reward = frames[idx]
    info_label.value = (
        f"<b>Step {step}/{len(frames)}</b> &nbsp;|&nbsp; "
        f"Action: {action} &nbsp;|&nbsp; "
        f"Bay: {selected_bay} &nbsp;|&nbsp; "
        f"Reward: {reward:.4f} &nbsp;|&nbsp; "
        f"Cumulative: {cum_reward:.4f}"
    )
    save_status.value = ""  # Clear save status on frame change
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(30, 16))
        ax.imshow(img)
        """
        ax.set_title(
            f"Seed {EVAL_SEED} | Step {step} | Action: {action} | Bay: {selected_bay} | "
            f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
        )
        """
        ax.axis("off")
        fig.tight_layout()
        plt.show()

def on_slider_change(change):
    show_frame(change["new"])

def on_prev(b):
    slider.value = max(0, slider.value - 1)

def on_next(b):
    slider.value = min(len(frames) - 1, slider.value + 1)

def on_save(b):
    save_current_frame(slider.value)

slider.observe(on_slider_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)

ui = widgets.VBox([
    info_label,
    widgets.HBox([prev_btn, slider, next_btn, save_btn]),
    save_status,
    out,
])

display(ui)
show_frame(0)

Run stacking simulator with Flat RL agent

In [ ]:
# ============================================================
# LOAD JOINT HIERARCHICAL MODEL + INTERACTIVE VISUAL EVALUATION
# (bay + stack trained jointly in a single model)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from stack.utils import mask_fn
from stack.models.joint_hierarchical_policy import MaskableJointTransformerPolicy
from stack.envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = config = {
            "vessel_shape": (10, 8, 5),
            "yard_shape": (10, 9, 5),
            "num_containers": 400,
            "group_num": 10,
            "group_placement": "random",
            "observation_type": "stack_features_v3",
            "reward_norm": False,
            "reward_clip": False,
            "stack_fill_penalty": True,
            "container_sizes": True,
            "enable_imo": True,
            "random_group_sizes": True
        }

JOINT_MODEL_PATH = "models/flat/pointer/large4/seed2/flat_pointer_large4_seed2/best_model" # ← update path
n_rows_per_bay = config_dict["yard_shape"][1]  # used to derive bay from stack action

# ============================================================
# CREATE ENV + LOAD MODEL
# ============================================================

eval_env = StackEnv(config=config_dict, render_mode="rgb_array")
eval_env = ActionMasker(eval_env, mask_fn)

model = MaskablePPO.load(
    JOINT_MODEL_PATH,
    #custom_objects={"policy_class": MaskableJointTransformerPolicy},
)

# ============================================================
# RUN EPISODE & COLLECT FRAMES
# ============================================================

obs, info = eval_env.reset(seed=30)
terminated = False
truncated = False
total_reward = 0.0
step_count = 0

frames = []  # (img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward)

while not (terminated or truncated) and step_count < 500:
    action, _ = model.predict(
        obs, deterministic=True, action_masks=get_action_masks(eval_env)
    )
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    step_count += 1

    img = eval_env.render()
    if img is not None:
        bay_idx = int(action) // n_rows_per_bay
        stack_in_bay = int(action) % n_rows_per_bay
        frames.append((img, step_count, int(action), bay_idx, stack_in_bay, reward, total_reward))

eval_env.close()

print(f"Episode finished: {len(frames)} frames | Total Reward: {total_reward:.4f}")

In [ ]:
# ============================================================
# INTERACTIVE VIEWER WIDGET (with Save Button)
# ============================================================

slider = widgets.IntSlider(
    value=0, min=0, max=len(frames) - 1, step=1,
    description="Step:", continuous_update=False,
    layout=widgets.Layout(width="60%")
)
prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
save_btn = widgets.Button(description="💾 Save", layout=widgets.Layout(width="80px"), button_style="info")
info_label = widgets.HTML()
out = widgets.Output()
save_status = widgets.HTML()

def save_current_frame(idx):
    img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward = frames[idx]
    import os
    os.makedirs("saved_eval_frames", exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(19, 13))
    ax.imshow(img)
    ax.set_title(
        f"Step {step} | Stack: {stack_action} | Bay: {bay_idx} | Stack-in-bay: {stack_in_bay} | "
        f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
    )
    ax.axis("off")
    fig.tight_layout()
    
    save_path = f"saved_eval_frames/hrl/episode_step_{step}.png"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    
    save_status.value = f'<span style="color:green;"><b>✓ Saved to {save_path}</b></span>'

def show_frame(idx):
    img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward = frames[idx]
    info_label.value = (
        f"<b>Step {step}/{len(frames)}</b> &nbsp;|&nbsp; "
        f"Stack Action: {stack_action} &nbsp;|&nbsp; "
        f"Bay: {bay_idx} &nbsp;|&nbsp; "
        f"Stack-in-bay: {stack_in_bay} &nbsp;|&nbsp; "
        f"Reward: {reward:.4f} &nbsp;|&nbsp; "
        f"Cumulative: {cum_reward:.4f}"
    )
    save_status.value = ""  # Clear save status on frame change
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(19, 13))
        ax.imshow(img)
        ax.set_title(
            f"Step {step} | Stack: {stack_action} | Bay: {bay_idx} | Stack-in-bay: {stack_in_bay} | "
            f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
        )
        ax.axis("off")
        fig.tight_layout()
        plt.show()

def on_slider_change(change):
    show_frame(change["new"])

def on_prev(b):
    slider.value = max(0, slider.value - 1)

def on_next(b):
    slider.value = min(len(frames) - 1, slider.value + 1)

def on_save(b):
    save_current_frame(slider.value)

slider.observe(on_slider_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)

ui = widgets.VBox([
    info_label,
    widgets.HBox([prev_btn, slider, next_btn, save_btn]),
    save_status,
    out
])

display(ui)
show_frame(0)

Run stacking simulator with Hierarchical RL agent

In [ ]:
# ============================================================
# LOAD JOINT HIERARCHICAL MODEL + INTERACTIVE VISUAL EVALUATION
# (bay + stack trained jointly in a single model)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from stack.utils import mask_fn
from stack.models.joint_hierarchical_policy import MaskableJointTransformerPolicy
from stack.envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = config = {
            "vessel_shape": (10, 8, 5),
            "yard_shape": (10, 9, 5),
            "num_containers": 400,
            "group_num": 10,
            "group_placement": "random",
            "observation_type": "stack_features_v3",
            "reward_norm": False,
            "reward_clip": False,
            "stack_fill_penalty": True,
            "container_sizes": True,
            "enable_imo": True,
            "random_group_sizes": True
        }

JOINT_MODEL_PATH = "models/hierarchical/pointer/large4/seed2/hrl_pointer_large4_seed2/best_model" # ← update path
n_rows_per_bay = config_dict["yard_shape"][1]  # used to derive bay from stack action

# ============================================================
# CREATE ENV + LOAD MODEL
# ============================================================

eval_env = StackEnv(config=config_dict, render_mode="rgb_array")
eval_env = ActionMasker(eval_env, mask_fn)

model = MaskablePPO.load(
    JOINT_MODEL_PATH,
    custom_objects={"policy_class": MaskableJointTransformerPolicy},
)

# ============================================================
# RUN EPISODE & COLLECT FRAMES
# ============================================================

obs, info = eval_env.reset(seed=30)
terminated = False
truncated = False
total_reward = 0.0
step_count = 0

frames = []  # (img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward)

while not (terminated or truncated) and step_count < 500:
    action, _ = model.predict(
        obs, deterministic=True, action_masks=get_action_masks(eval_env)
    )
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    step_count += 1

    img = eval_env.render()
    if img is not None:
        bay_idx = int(action) // n_rows_per_bay
        stack_in_bay = int(action) % n_rows_per_bay
        frames.append((img, step_count, int(action), bay_idx, stack_in_bay, reward, total_reward))

eval_env.close()

print(f"Episode finished: {len(frames)} frames | Total Reward: {total_reward:.4f}")

In [ ]:
# ============================================================
# INTERACTIVE VIEWER WIDGET (with Save Button)
# ============================================================

slider = widgets.IntSlider(
    value=0, min=0, max=len(frames) - 1, step=1,
    description="Step:", continuous_update=False,
    layout=widgets.Layout(width="60%")
)
prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
save_btn = widgets.Button(description="💾 Save", layout=widgets.Layout(width="80px"), button_style="info")
info_label = widgets.HTML()
out = widgets.Output()
save_status = widgets.HTML()

def save_current_frame(idx):
    img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward = frames[idx]
    import os
    os.makedirs("saved_eval_frames", exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(19, 13))
    ax.imshow(img)
    ax.set_title(
        f"Step {step} | Stack: {stack_action} | Bay: {bay_idx} | Stack-in-bay: {stack_in_bay} | "
        f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
    )
    ax.axis("off")
    fig.tight_layout()
    
    save_path = f"saved_eval_frames/hrl/episode_step_{step}.png"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    
    save_status.value = f'<span style="color:green;"><b>✓ Saved to {save_path}</b></span>'

def show_frame(idx):
    img, step, stack_action, bay_idx, stack_in_bay, reward, cum_reward = frames[idx]
    info_label.value = (
        f"<b>Step {step}/{len(frames)}</b> &nbsp;|&nbsp; "
        f"Stack Action: {stack_action} &nbsp;|&nbsp; "
        f"Bay: {bay_idx} &nbsp;|&nbsp; "
        f"Stack-in-bay: {stack_in_bay} &nbsp;|&nbsp; "
        f"Reward: {reward:.4f} &nbsp;|&nbsp; "
        f"Cumulative: {cum_reward:.4f}"
    )
    save_status.value = ""  # Clear save status on frame change
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(19, 13))
        ax.imshow(img)
        ax.set_title(
            f"Step {step} | Stack: {stack_action} | Bay: {bay_idx} | Stack-in-bay: {stack_in_bay} | "
            f"Reward: {reward:.4f} | Total: {cum_reward:.4f}"
        )
        ax.axis("off")
        fig.tight_layout()
        plt.show()

def on_slider_change(change):
    show_frame(change["new"])

def on_prev(b):
    slider.value = max(0, slider.value - 1)

def on_next(b):
    slider.value = min(len(frames) - 1, slider.value + 1)

def on_save(b):
    save_current_frame(slider.value)

slider.observe(on_slider_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)

ui = widgets.VBox([
    info_label,
    widgets.HBox([prev_btn, slider, next_btn, save_btn]),
    save_status,
    out
])

display(ui)
show_frame(0)